# Document Retrieval & Semantic Search Demo

This notebook demonstrates the document retrieval pipeline and semantic search capabilities.

**Goal**: Show how queries are matched to relevant documents using embeddings

In [ ]:
import sys
sys.path.insert(0, '../src')

from retrieval import RetrievalService
from ingestion import DataIngestionService
import pandas as pd

print("Modules imported successfully")

## 1. Initialize Services

In [ ]:
retrieval_service = RetrievalService()
ingestion_service = DataIngestionService()

doc_count = ingestion_service.get_document_count()
print(f"Documents available: {doc_count}")

if doc_count == 0:
    print("\n⚠️  No documents loaded. Run notebook 01_data_inspection first to load data.")
else:
    print("✓ Ready to retrieve documents")

## 2. Semantic Search - Single Query

In [ ]:
# Test a single query
query = "What is artificial intelligence?"

try:
    results = await retrieval_service.retrieve_documents(query=query, top_k=3)
    
    print(f"Query: {query}\n")
    print(f"Retrieved {len(results)} documents:\n")
    
    for i, result in enumerate(results, 1):
        print(f"\n--- Result {i} (Score: {result['score']:.4f}) ---")
        print(f"Content: {result['content'][:300]}...")
        if result['metadata']:
            print(f"Metadata: {result['metadata']}")
except Exception as e:
    print(f"Error retrieving documents: {e}")

## 3. Multi-Query Comparison

In [ ]:
# Test multiple queries to see different retrieval patterns
test_queries = [
    "machine learning algorithms",
    "neural networks",
    "deep learning applications",
    "computer vision",
    "natural language processing"
]

results_summary = []

for query in test_queries:
    try:
        results = await retrieval_service.retrieve_documents(query=query, top_k=3)
        top_doc = results[0] if results else None
        
        results_summary.append({
            'Query': query,
            'Top Score': top_doc['score'] if top_doc else 0,
            'Docs Retrieved': len(results),
            'Content Preview': top_doc['content'][:100] + "..." if top_doc else "N/A"
        })
    except Exception as e:
        print(f"Error with query '{query}': {e}")

# Display as table
df = pd.DataFrame(results_summary)
display(df)

## 4. Relevance Score Analysis

In [ ]:
# Analyze relevance scores across different top_k values
query = "artificial intelligence and machine learning"
top_k_values = [1, 3, 5, 10]

print(f"Query: {query}\n")

for top_k in top_k_values:
    try:
        results = await retrieval_service.retrieve_documents(query=query, top_k=top_k)
        scores = [r['score'] for r in results]
        
        print(f"\ntop_k={top_k}:")
        print(f"  Retrieved: {len(results)} documents")
        print(f"  Max score: {max(scores):.4f}")
        print(f"  Min score: {min(scores):.4f}")
        print(f"  Mean score: {sum(scores)/len(scores):.4f}")
        print(f"  Scores: {[f'{s:.4f}' for s in scores]}")
    except Exception as e:
        print(f"Error with top_k={top_k}: {e}")

## 5. Embedding Similarity Visualization

In [ ]:
# Show embedding similarity for related queries
query1 = "what is machine learning"
query2 = "what is deep learning"
query3 = "what is cooking"

try:
    embedding1 = retrieval_service.get_embedding(query1)
    embedding2 = retrieval_service.get_embedding(query2)
    embedding3 = retrieval_service.get_embedding(query3)
    
    # Calculate cosine similarity
    import numpy as np
    from numpy.linalg import norm
    
    def cosine_similarity(a, b):
        return np.dot(a, b) / (norm(a) * norm(b))
    
    sim_1_2 = cosine_similarity(embedding1, embedding2)
    sim_1_3 = cosine_similarity(embedding1, embedding3)
    
    print(f"Embedding Similarity:")
    print(f"\nQuery 1: {query1}")
    print(f"Query 2: {query2}")
    print(f"Similarity: {sim_1_2:.4f} (should be high - both about ML)\n")
    
    print(f"Query 1: {query1}")
    print(f"Query 3: {query3}")
    print(f"Similarity: {sim_1_3:.4f} (should be low - different topics)\n")
except Exception as e:
    print(f"Error calculating similarities: {e}")

## 6. Document Content Ranking

In [ ]:
# Show ranked documents 
query = "data science and analytics"

try:
    results = await retrieval_service.retrieve_documents(query=query, top_k=5)
    
    print(f"Query: {query}")
    print(f"\nTop {len(results)} documents ranked by relevance:\n")
    
    for rank, result in enumerate(results, 1):
        score_pct = result['score'] * 100
        bar = '█' * int(score_pct / 5)
        print(f"{rank}. [{bar:<20}] {result['score']:.4f} (100%)")
        print(f"   Content: {result['content'][:150]}...\n")
except Exception as e:
    print(f"Error: {e}")

## 7. Key Insights

In [ ]:
print("""
Key Observations:
================

1. **Relevance Scores**: Range from 0-1, where 1.0 is most relevant
   - Scores > 0.7 generally indicate good matches
   - Scores < 0.3 indicate weak relevance

2. **Document Ranking**: Results are ordered by relevance score (highest first)
   - Top documents are semantically closest to the query
   - Used as context for LLM answer generation

3. **top_k Parameter**: Controls how many documents to retrieve
   - Higher k = more context but potentially more noise
   - Typical range: 3-10 documents
   - Tradeoff between coverage and precision

4. **Embedding Quality**: Embeddings map text to high-dimensional vectors
   - Similar queries have similar embeddings
   - Enables semantic search beyond keyword matching

5. **Scalability**: System can handle thousands of documents efficiently
   - Vector search is O(log n) with proper indexing
   - Current index is suitable for millions of documents
""")